In [5]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [7]:
from app.core.config import settings
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg import AsyncConnection
from psycopg.rows import DictRow, dict_row
from psycopg_pool import AsyncConnectionPool

load_dotenv()
# 创建连接池
checkpoint_pool: AsyncConnectionPool[AsyncConnection[DictRow]] = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)

# 初始化连接
await checkpoint_pool.open()

# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()


In [8]:
# 初始化模型
model = init_chat_model(
    "deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

In [9]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "day06-checkpointer-test"}}

# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)

# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)

你今年30岁。你刚才在第一句话里告诉我了：“我叫小明，今年30岁。” 😊
